# Multi-Seed QA-FedAvg Run (GPU-Friendly)

This notebook runs **three sequential experiments** with the same `ALPHA` and `BETA` but different seeds (42, 123, 256). It is designed for Kaggle or Colab GPU runtimes.

Use four notebook copies in parallel by changing only `ALPHA` in Cell 1. Each notebook produces three seed runs and packages all artifacts for download.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Abishek-Chakravarthy/fed-rag.git"
REPO_BRANCH = "q-fedrag2"

# ── Change ALPHA per notebook copy ──
ALPHA = 0.0
BETA = 2.0

BETA = 2.0
SEEDS = [42, 123, 256]
BENCHMARK_MODE = "stress"
NOISE_MODE = "random_negative"
NOISE_RATIO = 0.8
ROUNDS = 4
LOCAL_EPOCHS = 1

ALLOWED_ALPHAS = {0.0, 0.3, 0.7, 1.0}
if ALPHA not in ALLOWED_ALPHAS:
    raise ValueError(f"Use one of {sorted(ALLOWED_ALPHAS)} for the parallel alpha notebooks")

WORKSPACE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
REPO_DIR = WORKSPACE / "fed-rag"
EXP_DIR = REPO_DIR / "zz_coderuns" / "quality_aware_fedrag" / "robustness_benchmark"
EXPORT_DIR = WORKSPACE / f"qa_fedavg_multiseed_exports_{BENCHMARK_MODE}"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"WORKSPACE : {WORKSPACE}")
print(f"REPO_DIR  : {REPO_DIR}")
print(f"EXP_DIR   : {EXP_DIR}")
print(f"TRACK     : {BENCHMARK_MODE}")
print(f"ALPHA     : {ALPHA}")
print(f"BETA      : {BETA}")
print(f"SEEDS     : {SEEDS}")

In [ ]:
import os
import shutil
import subprocess
import sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        REPO_BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ],
    check=True,
)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "protobuf>=4.25.3,<6",
    "accelerate",
    "datasets<3.0.0",
    "flwr==1.22.0",
    "pyarrow",
    "pydantic",
    "pydantic-settings",
    "transformers==4.48.0",
    "sentence-transformers==3.4.1",
    "peft",
    "matplotlib",
    "pandas",
    "tqdm",
], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"], check=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
print("Clone + install complete")


In [ ]:
import importlib.metadata
import subprocess
import torch

print("torch version:", torch.__version__)
print("protobuf version:", importlib.metadata.version("protobuf"))
print("cuda available:", torch.cuda.is_available())
print("mps available :", torch.backends.mps.is_available())

if torch.cuda.is_available():
    print("gpu device:", torch.cuda.get_device_name(0))
    subprocess.run(["nvidia-smi"])
elif torch.backends.mps.is_available():
    print("Using Apple Metal backend")
else:
    print("WARNING: No GPU backend detected. Switch the notebook runtime to GPU.")


In [ ]:
def build_run_slug(alpha: float, seed: int, benchmark_mode: str, noise_mode: str, noise_ratio: float, rounds: int, local_epochs: int) -> str:
    track = benchmark_mode.replace("-", "_")
    mode = noise_mode.replace("-", "_")
    ratio = f"{noise_ratio:.2f}".replace(".", "p")
    return f"alpha_{alpha:.1f}_track_{track}_seed_{seed}_mode_{mode}_ratio_{ratio}_r{rounds}_e{local_epochs}"

# Preview all run slugs.
for seed in SEEDS:
    slug = build_run_slug(ALPHA, seed, BENCHMARK_MODE, NOISE_MODE, NOISE_RATIO, ROUNDS, LOCAL_EPOCHS)
    print(slug)
print(f"\nExport root: {EXPORT_DIR}")

In [ ]:
import re
import subprocess
import sys
from tqdm.auto import tqdm

seed_results = {}

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*70}")
    print(f"  SEED RUN {seed_idx + 1}/{len(SEEDS)} — seed={seed}, α={ALPHA:.1f}, β={BETA}")
    print(f"{'='*70}\n")

    cmd = [
        sys.executable,
        "-u",
        "federated_noisy_qa.py",
        "--alpha", str(ALPHA),
        "--beta", str(BETA),
        "--seed", str(seed),
        "--benchmark-mode", BENCHMARK_MODE,
        "--noise-mode", NOISE_MODE,
        "--noise-ratio", str(NOISE_RATIO),
        "--rounds", str(ROUNDS),
        "--local-epochs", str(LOCAL_EPOCHS),
    ]

    print("Running:", " ".join(cmd))
    round_pattern = re.compile(r"Round\s+(\d+)\s+\|")
    progress = tqdm(total=ROUNDS, desc=f"seed={seed}", unit="round")

    process = subprocess.Popen(
        cmd,
        cwd=EXP_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        match = round_pattern.search(line)
        if match:
            progress.n = min(int(match.group(1)), ROUNDS)
            progress.refresh()

    process.wait()
    if process.returncode == 0:
        progress.n = ROUNDS
        progress.refresh()
    progress.close()

    if process.returncode != 0:
        print(f"ERROR: seed={seed} failed with returncode {process.returncode}")
        seed_results[seed] = {"status": "FAILED", "returncode": process.returncode}
        continue

    seed_results[seed] = {"status": "OK"}

print(f"\n{'='*70}")
print("  ALL SEED RUNS COMPLETE")
print(f"{'='*70}")
for s, info in seed_results.items():
    print(f"  seed={s}: {info['status']}")

In [ ]:
import json
import pandas as pd
import shutil
import zipfile

all_dfs = []

for seed in SEEDS:
    run_slug = build_run_slug(ALPHA, seed, BENCHMARK_MODE, NOISE_MODE, NOISE_RATIO, ROUNDS, LOCAL_EPOCHS)
    csv_path = EXP_DIR / "output_csv_files" / f"results_{run_slug}.csv"
    manifest_path = EXP_DIR / "output_csv_files" / f"manifest_{run_slug}.json"
    acceptance_path = EXP_DIR / "output_csv_files" / f"acceptance_{run_slug}.json"
    log_path = EXP_DIR / "output_log_files" / f"log_{run_slug}.log"

    print(f"\n--- seed={seed} ({run_slug}) ---")
    for path in [csv_path, manifest_path, acceptance_path, log_path]:
        print(f"  {path.name}: {path.exists()}")

    # Copy artifacts to export directory.
    run_export_dir = EXPORT_DIR / run_slug
    run_export_dir.mkdir(parents=True, exist_ok=True)
    for path in [csv_path, manifest_path, acceptance_path, log_path]:
        if path.exists():
            shutil.copy2(path, run_export_dir / path.name)

    # Read and display results.
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df["seed"] = seed
        all_dfs.append(df)
        print(f"  Final test MRR: {df['final_test_mrr'].iloc[-1]}")
        print(f"  Final test NDCG: {df['final_test_ndcg_at_k'].iloc[-1]}")
        print(f"  Best round: {df['selected_best_round'].iloc[-1]}")

    if acceptance_path.exists():
        with open(acceptance_path, "r", encoding="utf-8") as f:
            acceptance = json.load(f)
        print(f"  Acceptance:")
        for test_name, test_data in sorted(acceptance.items()):
            if isinstance(test_data, dict):
                print(f"    {test_name}: {test_data.get('passed', '?')}")

# Combine all seed results into one zip.
combined_zip = EXPORT_DIR / f"alpha_{ALPHA:.1f}_multiseed.zip"
with zipfile.ZipFile(combined_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for seed in SEEDS:
        run_slug = build_run_slug(ALPHA, seed, BENCHMARK_MODE, NOISE_MODE, NOISE_RATIO, ROUNDS, LOCAL_EPOCHS)
        run_export_dir = EXPORT_DIR / run_slug
        if run_export_dir.exists():
            for file_path in run_export_dir.iterdir():
                zf.write(file_path, arcname=f"{run_slug}/{file_path.name}")

print(f"\nCombined zip: {combined_zip}")

In [ ]:
import numpy as np

if all_dfs:
    combined = pd.concat(all_dfs, ignore_index=True)

    # Summary: one row per seed (last round = final test metrics).
    summary_rows = []
    for seed in SEEDS:
        seed_df = combined[combined["seed"] == seed]
        if seed_df.empty:
            continue
        last = seed_df.iloc[-1]
        summary_rows.append({
            "seed": seed,
            "alpha": ALPHA,
            "beta": BETA,
            "best_round": int(last["selected_best_round"]),
            "best_val_mrr": float(last["best_server_val_mrr_so_far"]),
            "final_test_mrr": float(last["final_test_mrr"]),
            "final_test_ndcg": float(last["final_test_ndcg_at_k"]),
            "final_test_recall": float(last["final_test_recall_at_k"]),
        })

    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)

    print(f"\n--- Multi-Seed Summary for α={ALPHA:.1f}, β={BETA} ---")
    print(f"  Mean Test MRR:  {summary_df['final_test_mrr'].mean():.5f} ± {summary_df['final_test_mrr'].std():.5f}")
    print(f"  Mean Test NDCG: {summary_df['final_test_ndcg'].mean():.5f} ± {summary_df['final_test_ndcg'].std():.5f}")
    print(f"  Seeds: {list(summary_df['seed'])}")
    print(f"  Best rounds: {list(summary_df['best_round'])}")
else:
    print("No results to summarize.")

## What to download

Run four notebook copies in parallel with `ALPHA = 0.0`, `0.3`, `0.7`, and `1.0`.

For each notebook, download:

- the `alpha_X.X_multiseed.zip` file, or
- the whole `qa_fedavg_multiseed_exports_mechanism/` folder

Each zip contains three sub-folders (one per seed), each with the CSV, manifest, acceptance, and log files.

Keep `BETA`, `BENCHMARK_MODE`, `ROUNDS`, and `LOCAL_EPOCHS` identical across all four notebooks.